Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Webhooks

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Webhooks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Preview: Webhooks support in the Gemini API is currently in preview.

Webhooks allow the Gemini API to push real-time notifications to your server
when asynchronous or Long-Running Operations (LROs) complete. This replaces the
need to poll the API for status updates, reducing latency and overhead.

Webhooks are available for operations like [Batch](https://ai.google.dev/gemini-api/docs/batch) jobs,
[Interactions](https://ai.google.dev/gemini-api/docs/interactions) and [video generation](https://ai.google.dev/gemini-api/docs/video).

## How it works

Instead of polling `GET /operations` repeatedly to check if a job is finished,
you can configure Gemini API Webhooks to send an HTTP POST request to your
listener URL immediately upon an event trigger.

The Gemini API supports two ways to configure webhooks:

* **Static webhooks**: Project-level endpoints. Good for global integrations (e.g., notifying Slack, syncing a
database, etc.).
* **Dynamic webhooks**: Request-level overrides passing a
webhook URL in the configuration payload of a specific jobs call. Ideal for
routing specific jobs to dedicated endpoints.


# Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q "google-genai>=1.73.1" # Minimum version 1.73.1 for webhooks, if you see an error related to google-auth version mismatch, re-run the cell

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

Initialize a client with your API key:

In [ ]:
import os
from google import genai
from google.genai import errors, types

client = genai.Client(api_key=GEMINI_API_KEY)

## Step 1: Create a webhook endpoint

This example uses a Google Cloud Run function, which allow you to easily deploy a serverless function you can use as our webhook endpoint.

1. Go to https://console.cloud.google.com/run/
2. Click NodeJS under Write a function
3. Enter a name such as "gemini-webhook-receiver" and select "Allow public access" under Authentication
4. Click "Deploy". If this is the first time you're deploying a Cloud Run function, you may need to enable APIs such as Cloud Build as prompted.

(Their management UI at https://console.cloud.google.com/run/ has a log observation page, you can see the requests there)

5. **Implement webhook signature verification**: In the Source tab, update the `package.json` dependencies with

```jsdo
"dependencies": {
		"standardwebhooks": "^1.0.0"
	}
```

Update the Function entry point to `geminiWebhook` 

and replace the code in `/src/index.js` with the following code, then click "Save abd redeploy"

```javascript
const functions = require('@google-cloud/functions-framework');
const { Webhook } = require('standardwebhooks');

/**
 * Google Cloud Function to handle Gemini API Webhooks
 */
functions.http('geminiWebhook', async (req, res) => {
  // Respond with a 200 for non-POST requests to indicate that the function is running
  if (req.method !== 'POST') {
    return res.status(200).send('Worker is running');
  }

  // Print headers and body to stdout
  console.log('--- Webhook Headers ---');
  console.log(JSON.stringify(req.headers, null, 2));
  console.log('--- Webhook Body ---');
  console.log(JSON.stringify(req.body, null, 2));

  const payload = JSON.stringify(req.body);
  
  // Extract specific headers for Standard Webhooks verification
  // This avoids issues with extra headers added by Cloud Run/Functions proxy
  const headers = {
    'webhook-id': req.headers['webhook-id'],
    'webhook-signature': req.headers['webhook-signature'],
    'webhook-timestamp': req.headers['webhook-timestamp'],
  };

  const secret = process.env.WEBHOOK_SIGNING_SECRET;

  try {
    const wh = new Webhook(secret);
    // Verify the webhook signature using only the relevant headers
    const event = wh.verify(payload, headers);
    
    console.log(`Event type: ${event.type}, data: ${JSON.stringify(event.data)}`);

    if (event.type === 'batch.succeeded') {
      console.log(`Batch completed! ID: ${event.data.id}`);
      if (event.data.output_file_uri) {
        console.log(`Batch file: ${event.data.output_file_uri}`);
      }
    } else if (event.type === 'video.generated') {
      console.log(`Video generated! URI: ${event.data.video_uri}`);
    }

    res.status(200).json({ status: 'received' });
  } catch (err) {
    console.error('Webhook verification failed:', err.message);
    res.status(400).json({ error: 'Signature invalid' });
  }
});
```

 When an event happens that you're subscribed to, your webhook URL will receive an HTTP POST request. Your endpoint must respond with a 2xx status code within a few seconds to avoid a retry. To ensure delivery, the Gemini API automatically retries failed requests for 24 hours using exponential backoff.


TODO: Change this to use your own webhook endpoint!

In [ ]:
# TODO: Copy and paste the URL for the function you just deployed above
WEBHOOK_ENDPOINT="https://your-app.run.app"  # @param {type:"string"}

Check that the endpoint is working. This example endpoint simply returns a text for a GET request.

In [ ]:
import requests

r = requests.get(WEBHOOK_ENDPOINT)
print(r.text)

Worker is running!


## Step 2: Create a webhook and add the Secret to Cloud Run function

**IMPORTANT**: When creating a webhook, the API returns a **signing secret**
**only once**. Take the secret below and add it to your Cloud Run function.

Steps
- On Google Cloud Console for your function, Click "Edit and deploy new revision"
- Then, in the "Containers" > "Variables and Secrets" tab, click "Add variable"
- Then, enter the secret "WEBHOOK_SIGNING_SECRET" with the value as follows.
- Then click Deploy.

In [ ]:
webhook = client.webhooks.create(
    subscribed_events=["video.generated"],
    name="video_generation_webhook",
    uri=WEBHOOK_ENDPOINT,
)

# Store webhook.signing_secret securely (e.g., in your env vars)
os.environ["WEBHOOK_SIGNING_SECRET"] = webhook.new_signing_secret

# print(f"WEBHOOK_SIGNING_SECRET={webhook.new_signing_secret}")
# # Or write it to a file:
# with open(".env", "a") as f:
#     f.write(f"WEBHOOK_SIGNING_SECRET={webhook.new_signing_secret}\n")

print(f"Created webhook: {webhook.name}")


Created webhook: video_generation_webhook


### List

List all configured webhooks for the current project, with optional pagination.

In [ ]:
client.webhooks.list()

WebhookListResponse(next_page_token=None, webhooks=[Webhook(subscribed_events=['video.generated'], uri='https://read-post.pythonengineerman.workers.dev/', id='k0h9q38f6ejj1g9j7cgcdsauadmbewegl79f', create_time=None, name='video_generation_webhook', new_signing_secret=None, signing_secrets=[SigningSecret(expire_time=None, truncated_secret='whsec_...3frh')], state='enabled', update_time=None)])

### Get/Ping/Delete/Update

Ping a webhook

In [ ]:
client.webhooks.ping(id=webhook.id)

WebhookPingResponse()

Get a webhook

In [ ]:
wh = client.webhooks.get(id=webhook.id)
wh

This would delete the webhook. Set `delete=True` to run it:

In [ ]:
delete = False
if delete:
    client.webhooks.delete(id=webhook.id)

This is how you can update a webhook. Set `update=True` to run it:

In [ ]:
update = False
if update:
    wh = client.webhooks.update(
        id=webhook.id,
        subscribed_events=["video.generated", "batch.succeeded"],
        name="video_and_batch_webhook",
    )

## Step 4: Send a Gemini API request

Send a request for the long running operation your webhook is subscribed to.

In [ ]:
prompt = """A stunning drone view of the Grand Canyon during a flamboyant sunset that highlights the canyon's colors. The drone slowly flies towards the sun then accelerates, dives and flies inside the canyon."""

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
)

Now, instead of polling GET /operations, your webhook should receive an HTTP POST request once the video generation has finished.

## Step 5: Check the worker logs

Check the **Observability > Logs** tab in your Cloud Run Dashboard 

It should have logs that show the Webhook payload and the output of the function, in this case it should display the URL of the generated video.

## Dynamic webhook config

You can also use dynamic webhook configuration, but for this, you need to update your webhook verification.

See the docs for implementation details:

- https://ai.google.dev/gemini-api/docs/webhooks#dynamic-webhooks
- https://ai.google.dev/gemini-api/docs/webhooks#verify-dynamic-signatures


## Next Steps:

- **Webhooks documentation**: Read the full [Webhooks guide](https://ai.google.dev/gemini-api/docs/webhooks) for advanced configuration options, supported event types, and best practices.
- **Batch API**: Webhooks pair well with the [Batch API](./Batch_mode.ipynb) to process large volumes of requests and get notified when they complete.
- **Video generation**: Use webhooks with [Veo](./Get_started_Veo.ipynb) to get notified when your generated videos are ready.
- **Cloud Run functions**: Learn more about [Cloud Run functions](https://cloud.google.com/functions).
- **Standard Webhooks**: The Gemini API uses the [Standard Webhooks](https://www.standardwebhooks.com/) specification for signature verification, ensuring secure and interoperable webhook delivery.
